# Matrix Calculus Cheatsheet — Exercises

Companion to the note [Matrix Calculus Cheatsheet](Matrix%20Calculus%20Cheatsheet.md).

Practise deriving every row of the cheatsheet (denominator layout, gradients as column vectors), the vector chain rule $\nabla_xz = J_h(x)^\top\nabla_yz$, and the common derivatives (sigmoid, softmax + cross-entropy, tanh, ReLU). Then verify everything numerically with finite differences and hand-written backprop in NumPy.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×8 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit, softmax
from sklearn.linear_model import Ridge
rng = np.random.default_rng(0)

def fd_grad(f, x, h=1e-6):
    # reference central-difference gradient for arrays of any shape (used by the checks)
    x = np.asarray(x, float); g = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        e = np.zeros_like(x); e[idx] = h
        g[idx] = (f(x + e) - f(x - e)) / (2 * h)
    return g

## Part A · Concepts

### Exercise 1 · Layouts and shapes
*🧠 Concept · ★☆☆*

In denominator layout (the note's convention), what is the shape of: (a) $\nabla_x f$ for $f:\mathbb R^n\to\mathbb R$;
(b) $\nabla_X f$ for $f:\mathbb R^{m\times n}\to\mathbb R$; (c) the Jacobian $J_h(x)$ of $h:\mathbb R^n\to\mathbb R^m$ (standard
definition $(J_h)_{ij}=\partial h_i/\partial x_j$)? Why do the chain-rule formulas of different textbooks look transposed relative to each other?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Denominator layout: the gradient has the shape of the variable you differentiate with respect to.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $n\times1$, a column vector like $x$. (b) $m\times n$, like $X$. (c) $m\times n$.
Numerator layout writes $\partial f/\partial x$ as a $1\times n$ row (the Jacobian of a scalar function), denominator layout as its
transpose. The same chain rule therefore appears as $J_g J_h$ in one book and $J_h^\top\nabla_y z$ in another. The fix is to
pick one convention and **always check shapes**: a gradient must have the shape of its variable.

</details>

### Exercise 2 · Shape checking catches bugs
*🧠 Concept · ★★☆*

A student claims $\nabla_x\lVert Ax-b\rVert^2 = 2(Ax-b)A$ for $A\in\mathbb R^{m\times n}$. Without differentiating, show why
this cannot be right, and find the only shape-consistent arrangement of the factors $2$, $A$ and $(Ax-b)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$(Ax-b)$ is $m\times1$; the gradient must be $n\times1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(Ax-b)A$ multiplies an $m\times1$ by an $m\times n$ matrix: undefined (unless $m=1$). The result must be $n\times1$, and the only
way to combine an $m\times n$ matrix with an $m\times1$ vector into an $n\times1$ vector is $A^\top(Ax-b)$. So
$\nabla_x = 2A^\top(Ax-b)$, the cheatsheet entry. Shape analysis does not prove the formula, but it fixes the order of the factors and transposes.

</details>

### Exercise 3 · Why backpropagation runs backwards
*🧠 Concept · ★★☆*

For $L = f_3(f_2(f_1(x)))$ with $x\in\mathbb R^n$, intermediate sizes $n$, and scalar $L$, the gradient is
$\nabla_xL = J_1^\top J_2^\top\nabla_{y_3}L$ ($J_k$ are $n\times n$). Compare the cost of evaluating this product
right-to-left (reverse mode) versus left-to-right (forward mode). Why is reverse mode the right choice for training neural
networks, and what is its memory cost? ([[Backpropagation]])

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Matrix–vector products cost $O(n^2)$, matrix–matrix products $O(n^3)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Right-to-left: $J_2^\top\nabla L$ is a matrix–vector product, then $J_1^\top(\cdot)$ again, so $O(n^2)$ per layer. Left-to-right
forms $J_1^\top J_2^\top$ first: $O(n^3)$ matrix–matrix products. With one scalar output and millions of parameters, reverse
mode gives the full gradient for about the cost of one extra forward pass. In practice the Jacobians are never built; each layer
implements a vector–Jacobian product. The price: all forward activations must be stored for the backward pass (memory
grows with depth; gradient checkpointing trades compute for memory).

</details>

## Part B · By hand

### Exercise 4 · Linear and quadratic forms
*✍️ By hand · ★☆☆*

Derive $\nabla_x(a^\top x) = a$ and $\nabla_x(x^\top Ax) = (A+A^\top)x$ in index notation. Then evaluate the second at
$A=\begin{pmatrix}1&2\\0&3\end{pmatrix}$, $x=(1,-1)^\top$. Why is it **not** $2Ax$ here?

In [ ]:
A_q = np.array([[1., 2.], [0., 3.]]); x_q = np.array([1., -1.])
grad_quad = None

check('grad of x^T A x', grad_quad, fd_grad(lambda x: x @ A_q @ x, x_q))

<details>
<summary><b>💡 Hint</b></summary>

$x^\top Ax = \sum_{i,j}x_iA_{ij}x_j$. Differentiate w.r.t. $x_k$: the terms with $i=k$ and those with $j=k$ both contribute.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial_k\sum_ia_ix_i = a_k$. $\partial_k\sum_{ij}x_iA_{ij}x_j = \sum_jA_{kj}x_j + \sum_ix_iA_{ik} = (Ax)_k + (A^\top x)_k$.
Here $A+A^\top = \begin{pmatrix}2&2\\2&6\end{pmatrix}$, so the gradient is $(2-2,\ 2-6) = (0,-4)$. $A$ is not symmetric, so $2Ax=(-2,-6)$ would be wrong.

```python
A_q = np.array([[1., 2.], [0., 3.]]); x_q = np.array([1., -1.])
grad_quad = [0., -4.]
```

</details>

### Exercise 5 · Least-squares gradient
*✍️ By hand · ★☆☆*

Use the previous result to derive $\nabla_x\lVert Ax-b\rVert^2 = 2A^\top(Ax-b)$ by expanding the square. Evaluate it for
$A=\begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$, $b=(1,2,2)^\top$, $x=(1,1)^\top$.

In [ ]:
A_ls = np.array([[1., 0.], [1., 1.], [1., 2.]]); b_ls = np.array([1., 2., 2.]); x_ls = np.array([1., 1.])
grad_ls = None

check('grad of ||Ax-b||^2', grad_ls, fd_grad(lambda x: np.sum((A_ls @ x - b_ls) ** 2), x_ls))

<details>
<summary><b>💡 Hint</b></summary>

$\lVert Ax-b\rVert^2 = x^\top A^\top Ax - 2b^\top Ax + b^\top b$; $A^\top A$ is symmetric.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla = 2A^\top Ax - 2A^\top b = 2A^\top(Ax-b)$. Here $Ax = (1,2,3)$, residual $(0,0,1)$, $A^\top r = (1,2)$, gradient $(2,4)$.
Setting it to zero gives the normal equations ([[Linear Regression]]).

```python
A_ls = np.array([[1., 0.], [1., 1.], [1., 2.]]); b_ls = np.array([1., 2., 2.]); x_ls = np.array([1., 1.])
grad_ls = [2., 4.]
```

</details>

### Exercise 6 · Sigmoid, tanh and ReLU derivatives
*✍️ By hand · ★☆☆*

Show $\sigma'(z)=\sigma(z)(1-\sigma(z))$ for $\sigma(z)=1/(1+e^{-z})$ and $\tanh'(z)=1-\tanh^2 z$. Compute $\sigma'(0)$, $\sigma'(2)$ and
$\tanh'(1)$. What is the maximum of $\sigma'$, and what does that imply for deep sigmoid networks?

In [ ]:
dsig0 = None
dsig2 = None
dtanh1 = None

check("sigma'(0)", dsig0, fd_grad(lambda z: expit(z), np.array(0.0)), tol=1e-6)
check("sigma'(2)", dsig2, fd_grad(lambda z: expit(z), np.array(2.0)), tol=1e-6)
check("tanh'(1)", dtanh1, fd_grad(np.tanh, np.array(1.0)), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}}$ and $1-\sigma(z)=\frac{e^{-z}}{1+e^{-z}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sigma'(z)=\sigma(z)(1-\sigma(z))$ as in the hint; $\tanh = (e^z-e^{-z})/(e^z+e^{-z})$ and the quotient rule give $1-\tanh^2$.
$\sigma'(0)=0.25$, $\sigma'(2)=0.8808\cdot0.1192\approx0.1050$, $\tanh'(1) = 1-0.7616^2\approx0.4200$.
$\max\sigma' = 1/4$: backprop multiplies by at most $1/4$ per sigmoid layer, so gradients vanish geometrically with depth. ReLU's
derivative is $1$ on the active side, one reason it replaced sigmoids.

```python
dsig0 = 0.25
dsig2 = expit(2) * (1 - expit(2))
dtanh1 = 1 - np.tanh(1) ** 2
```

</details>

### Exercise 7 · Softmax + cross-entropy: $\hat p - y$
*✍️ By hand · ★★☆*

With $\hat p = \mathrm{softmax}(z)$ and $L = -\sum_ky_k\log\hat p_k$ for one-hot $y$, show $\partial L/\partial z = \hat p - y$.
Evaluate it for $z = (1, 2, 3)$ with true class $0$.

In [ ]:
grad_ce = None

check('dL/dz', grad_ce, fd_grad(lambda z: -np.log(softmax(z)[0]), np.array([1., 2., 3.])))

<details>
<summary><b>💡 Hint</b></summary>

$L = -z_c + \log\sum_je^{z_j}$ for true class $c$. Differentiate both terms w.r.t. $z_k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial_k L = -\mathbb 1[k=c] + \frac{e^{z_k}}{\sum_je^{z_j}} = \hat p_k - y_k$. For $z=(1,2,3)$: $\hat p\approx(0.0900, 0.2447, 0.6652)$, so
$\nabla_zL \approx (-0.9100,\ 0.2447,\ 0.6652)$. The same form ($\hat p - y$) appears for sigmoid + binary cross-entropy: the output
layer's error signal is simply prediction minus target.

```python
grad_ce = softmax(np.array([1., 2., 3.])) - np.array([1., 0., 0.])
```

</details>

### Exercise 8 · Gradient of a log-determinant
*✍️ By hand · ★★★*

Using Jacobi's formula $d\,|A| = |A|\,\mathrm{tr}(A^{-1}dA)$, show that $\nabla_A\ln|A| = A^{-\top}$. Evaluate it for
$A=\begin{pmatrix}2&1\\0&3\end{pmatrix}$. Where does this gradient appear in ML?

In [ ]:
A_det = np.array([[2., 1.], [0., 3.]])
grad_logdet = None   # 2x2 nested list

check('grad ln|A|', grad_logdet, fd_grad(lambda A: np.log(np.linalg.det(A)), A_det))

<details>
<summary><b>💡 Hint 1</b></summary>

$d\ln|A| = \mathrm{tr}(A^{-1}dA)$. Match it to $d f = \sum_{ij}G_{ij}dA_{ij} = \mathrm{tr}(G^\top dA)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For the numeric part: $A^{-1} = \frac16\begin{pmatrix}3&-1\\0&2\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$d\ln|A| = \mathrm{tr}(A^{-1}dA) = \mathrm{tr}(G^\top dA)$ with $G = A^{-\top}$. Numerically
$A^{-\top} = \begin{pmatrix}1/2&0\\-1/6&1/3\end{pmatrix}$. It appears when maximising a Gaussian likelihood w.r.t. $\Sigma$ (MLE of
covariance, [[Gaussian Mixture Models and EM]]) and in the log-determinant of normalising flows.

```python
A_det = np.array([[2., 1.], [0., 3.]])
grad_logdet = [[0.5, 0.], [-1 / 6, 1 / 3]]
```

</details>

### Exercise 9 · Trace gradients
*✍️ By hand · ★★☆*

Show (i) $\nabla_X\mathrm{tr}(AX) = A^\top$ and (ii) $\nabla_X\mathrm{tr}(X^\top BX) = (B+B^\top)X$ using index notation. As a check,
note that (ii) reduces to the quadratic-form row of the cheatsheet when $X$ is a single column.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{tr}(AX) = \sum_{i,j}A_{ij}X_{ji}$, so $\partial/\partial X_{kl}$ picks the term with $j=k, i=l$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) $\partial_{X_{kl}}\sum_{ij}A_{ij}X_{ji} = A_{lk}$, i.e. the gradient is $A^\top$.
(ii) $\mathrm{tr}(X^\top BX) = \sum_c x_c^\top Bx_c$ over the columns $x_c$ of $X$, so each column's gradient is $(B+B^\top)x_c$,
stacked: $(B+B^\top)X$. For a single column this is exactly $\nabla_x x^\top Bx$.

</details>

### Exercise 10 · Chain rule through a layer
*✍️ By hand · ★★☆*

Let $y = Wx$ ($W\in\mathbb R^{m\times n}$), $h=\tanh(y)$ (elementwise) and $z = \lVert h\rVert^2$. Use the chain rule
$\nabla_xz = J^\top\nabla z$ step by step to derive $\nabla_xz$ and $\nabla_Wz$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\nabla_hz = 2h$; the Jacobian of an elementwise function is diagonal: $J = \mathrm{diag}(1-h^2)$; and $J_{y}(x) = W$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_h z = 2h$. Elementwise tanh: $\nabla_yz = \mathrm{diag}(1-h^2)\,2h = 2h\odot(1-h^2) =: \delta$.
Linear layer: $\nabla_xz = W^\top\delta$ and $\nabla_Wz = \delta x^\top$ (since $\partial y_i/\partial W_{ij} = x_j$).
This is the pattern of every backprop step: multiply elementwise by the activation derivative, then by $W^\top$ to go down a layer,
and take an outer product with the layer input for the weight gradient.

</details>

### Exercise 11 · Gaussian MLE via matrix calculus
*✍️ By hand · ★★★*

For i.i.d. $x_i\sim\mathcal N(\mu,\Sigma)$, $\ell(\mu,\Sigma) = -\frac N2\ln|\Sigma| - \frac12\sum_i(x_i-\mu)^\top\Sigma^{-1}(x_i-\mu) + \text{const}$.
Derive $\hat\mu$ from $\nabla_\mu\ell=0$. For $\Sigma$, differentiate w.r.t. the precision $\Lambda=\Sigma^{-1}$ (using
$\ln|\Sigma| = -\ln|\Lambda|$ and the log-det and trace rows of the cheatsheet) to get $\hat\Sigma$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$(x-\mu)^\top\Lambda(x-\mu) = \mathrm{tr}\big(\Lambda(x-\mu)(x-\mu)^\top\big)$ and $\nabla_\Lambda\mathrm{tr}(\Lambda S) = S^\top = S$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_\mu\ell = \Sigma^{-1}\sum_i(x_i-\mu) = 0\Rightarrow\hat\mu = \frac1N\sum_ix_i$.
With $S=\sum_i(x_i-\mu)(x_i-\mu)^\top$: $\ell = \frac N2\ln|\Lambda| - \frac12\mathrm{tr}(\Lambda S)$, so
$\nabla_\Lambda\ell = \frac N2\Lambda^{-\top} - \frac12S = 0\Rightarrow\Lambda^{-1} = \hat\Sigma = \frac1N\sum_i(x_i-\hat\mu)(x_i-\hat\mu)^\top$
(the biased sample covariance, see [[Probability for ML]]).

</details>

## Part C · Verify numerically

### Exercise 12 · A finite-difference gradient checker
*💻 Code · ★☆☆*

Write `num_grad(f, x, h=1e-6)` (central differences, works for any array shape) and use it to verify the first four rows of the
cheatsheet on random $a, A, b, x$. Store the four booleans (analytic ≈ numeric) in `rows_ok`.

In [ ]:
def num_grad(f, x, h=1e-6):
    return None

a_r = rng.normal(size=4); A_r = rng.normal(size=(4, 4)); b_r = rng.normal(size=4); x_r = rng.normal(size=4)
rows_ok = None

check('num_grad on a quadratic', num_grad(lambda x: x @ A_r @ x, x_r), (A_r + A_r.T) @ x_r, tol=1e-5)
check('all four rows verified', None if rows_ok is None else all(rows_ok), True)

<details>
<summary><b>💡 Hint</b></summary>

Loop over `np.ndindex(x.shape)`, perturb one entry by $\pm h$ and divide the difference by $2h$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Central differences have truncation error $O(h^2)$; with float64, $h\approx10^{-5}$–$10^{-6}$ balances truncation and round-off. Always compare with a *relative* tolerance.

```python
def num_grad(f, x, h=1e-6):
    x = np.asarray(x, float); g = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        e = np.zeros_like(x); e[idx] = h
        g[idx] = (f(x + e) - f(x - e)) / (2 * h)
    return g

a_r = rng.normal(size=4); A_r = rng.normal(size=(4, 4)); b_r = rng.normal(size=4); x_r = rng.normal(size=4)
rows_ok = [
    np.allclose(num_grad(lambda x: a_r @ x, x_r), a_r, atol=1e-6),
    np.allclose(num_grad(lambda x: x @ A_r @ x, x_r), (A_r + A_r.T) @ x_r, atol=1e-5),
    np.allclose(num_grad(lambda x: x @ x, x_r), 2 * x_r, atol=1e-6),
    np.allclose(num_grad(lambda x: np.sum((A_r @ x - b_r) ** 2), x_r), 2 * A_r.T @ (A_r @ x_r - b_r), atol=1e-5),
]
print(rows_ok)
```

</details>

### Exercise 13 · Matrix-valued gradients
*💻 Code · ★★☆*

Write analytic functions `grad_logdet(A)` ($=A^{-\top}$) and `grad_trAX(A, X)` ($=A^\top$), plus `grad_trXtBX(B, X)`
($=(B+B^\top)X$), and verify them with your `num_grad` on random matrices (make $A$ well-conditioned with positive determinant,
e.g. $A = I\cdot 3 + 0.5\,\text{noise}$).

In [ ]:
def grad_logdet(A):
    return None
def grad_trAX(A, X):
    return None
def grad_trXtBX(B, X):
    return None

A_m = 3 * np.eye(3) + 0.5 * rng.normal(size=(3, 3)); X_m = rng.normal(size=(3, 3)); B_m = rng.normal(size=(3, 3))

check('grad ln|A|', grad_logdet(A_m), fd_grad(lambda A: np.log(np.linalg.det(A)), A_m), tol=1e-5)
check('grad tr(AX)', grad_trAX(A_m, X_m), fd_grad(lambda X: np.trace(A_m @ X), X_m), tol=1e-5)
check('grad tr(X^T B X)', grad_trXtBX(B_m, X_m), fd_grad(lambda X: np.trace(X.T @ B_m @ X), X_m), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.inv(A).T`, `A.T`, `(B + B.T) @ X`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Numerically checking matrix gradients is the same as for vectors: perturb one entry at a time. The checks above use a reference finite-difference routine; yours should agree to about $10^{-8}$.

```python
def grad_logdet(A):
    return np.linalg.inv(A).T
def grad_trAX(A, X):
    return A.T
def grad_trXtBX(B, X):
    return (B + B.T) @ X

A_m = 3 * np.eye(3) + 0.5 * rng.normal(size=(3, 3)); X_m = rng.normal(size=(3, 3)); B_m = rng.normal(size=(3, 3))
print(np.abs(grad_logdet(A_m) - num_grad(lambda A: np.log(np.linalg.det(A)), A_m)).max())
```

</details>

### Exercise 14 · Batch softmax cross-entropy gradient
*💻 Code · ★★☆*

For logits $Z = XW$ ($X$: $N\times d$, $W$: $d\times K$) and mean cross-entropy $L(W)=\frac1N\sum_i -\log\hat p_{i,y_i}$, derive and
implement `ce_grad_W(W, X, y)` $= \frac1N X^\top(\hat P - Y)$ where $Y$ is one-hot. Verify against finite differences.

In [ ]:
def ce_loss_W(W, X, y):
    Z = X @ W
    Z = Z - Z.max(1, keepdims=True)
    return np.mean(np.log(np.exp(Z).sum(1)) - Z[np.arange(len(y)), y])

def ce_grad_W(W, X, y):
    return None

X_s = rng.normal(size=(30, 5)); y_s = rng.integers(0, 3, 30); W_s = rng.normal(size=(5, 3))

check('dL/dW', ce_grad_W(W_s, X_s, y_s), fd_grad(lambda W: ce_loss_W(W, X_s, y_s), W_s), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Per sample $\partial L/\partial z_i = \hat p_i - y_i$ (Exercise 7); then $Z=XW$ is a linear layer, so $\nabla_W = X^\top\nabla_Z$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Stacking the per-sample $\hat p_i - y_i$ as rows of $\nabla_Z = (\hat P-Y)/N$ and applying the linear-layer rule gives $X^\top(\hat P-Y)/N$, the gradient of multinomial logistic regression ([[Logistic Regression]]).

```python
def ce_loss_W(W, X, y):
    Z = X @ W
    Z = Z - Z.max(1, keepdims=True)
    return np.mean(np.log(np.exp(Z).sum(1)) - Z[np.arange(len(y)), y])

def ce_grad_W(W, X, y):
    P = softmax(X @ W, axis=1)
    Y = np.eye(W.shape[1])[y]
    return X.T @ (P - Y) / len(y)

X_s = rng.normal(size=(30, 5)); y_s = rng.integers(0, 3, 30); W_s = rng.normal(size=(5, 3))
```

</details>

### Exercise 15 · Backprop for a two-layer network
*💻 Code · ★★★*

For one input $x\in\mathbb R^3$, target $t\in\mathbb R^2$ and
$$h = \tanh(W_1x),\qquad \hat y = W_2h,\qquad L = \tfrac12\lVert\hat y - t\rVert^2,$$
implement `backprop(W1, W2, x, t)` returning `(dW1, dW2)` using only the chain rule (no finite differences). Check against
finite differences.

In [ ]:
def backprop(W1, W2, x, t):
    # TODO: forward pass, then backward pass; return dW1, dW2
    return None, None

W1_n = rng.normal(size=(4, 3)); W2_n = rng.normal(size=(2, 4)); x_n = rng.normal(size=3); t_n = rng.normal(size=2)
dW1, dW2 = backprop(W1_n, W2_n, x_n, t_n)

_L = lambda W1, W2: 0.5 * np.sum((W2 @ np.tanh(W1 @ x_n) - t_n) ** 2)
check('dL/dW2', dW2, fd_grad(lambda W: _L(W1_n, W), W2_n))
check('dL/dW1', dW1, fd_grad(lambda W: _L(W, W2_n), W1_n))

<details>
<summary><b>💡 Hint 1</b></summary>

$\delta_2 = \hat y - t$; $\nabla_{W_2} = \delta_2h^\top$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\delta_1 = (W_2^\top\delta_2)\odot(1-h^2)$; $\nabla_{W_1} = \delta_1x^\top$ (Exercise 10).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Exactly the pattern from Exercise 10: error signal at the output, propagate down with $W^\top$, multiply by the activation derivative, then outer product with the layer input. Autograd libraries automate this bookkeeping ([[Backpropagation]], [[Neural Networks]]).

```python
def backprop(W1, W2, x, t):
    h = np.tanh(W1 @ x); yhat = W2 @ h
    d2 = yhat - t
    dW2 = np.outer(d2, h)
    d1 = (W2.T @ d2) * (1 - h ** 2)
    dW1 = np.outer(d1, x)
    return dW1, dW2

W1_n = rng.normal(size=(4, 3)); W2_n = rng.normal(size=(2, 4)); x_n = rng.normal(size=3); t_n = rng.normal(size=2)
dW1, dW2 = backprop(W1_n, W2_n, x_n, t_n)
```

</details>

### Exercise 16 · Ridge regression: gradient to closed form
*💻 Code · ★☆☆*

For $L(w) = \lVert Xw-y\rVert^2 + \lambda\lVert w\rVert^2$, write `ridge_grad(w, X, y, lam)` using the cheatsheet, then set the
gradient to zero and implement `ridge_closed(X, y, lam)`. Compare with `Ridge(alpha=lam, fit_intercept=False)`.

In [ ]:
def ridge_grad(w, X, y, lam):
    return None
def ridge_closed(X, y, lam):
    return None

X_rd = rng.normal(size=(40, 5)); y_rd = rng.normal(size=40); w_rd = rng.normal(size=5)

check('ridge gradient', ridge_grad(w_rd, X_rd, y_rd, 2.0), fd_grad(lambda w: np.sum((X_rd @ w - y_rd) ** 2) + 2.0 * w @ w, w_rd), tol=1e-5)
check('ridge closed form', ridge_closed(X_rd, y_rd, 2.0), Ridge(alpha=2.0, fit_intercept=False).fit(X_rd, y_rd).coef_)

<details>
<summary><b>💡 Hint</b></summary>

$\nabla L = 2X^\top(Xw-y) + 2\lambda w = 0\Rightarrow(X^\top X+\lambda I)w = X^\top y$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Two cheatsheet rows ($\lVert Ax-b\rVert^2$ and $\lVert x\rVert^2$) give the gradient; its zero is a linear system. Use `np.linalg.solve`, not `inv`.

```python
def ridge_grad(w, X, y, lam):
    return 2 * X.T @ (X @ w - y) + 2 * lam * w
def ridge_closed(X, y, lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

X_rd = rng.normal(size=(40, 5)); y_rd = rng.normal(size=40); w_rd = rng.normal(size=5)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Matrix Calculus Cheatsheet](Matrix%20Calculus%20Cheatsheet.md).*